In [1]:
import numpy as np
import xarray as xr
import toml
import munch
from tqdm import tqdm
import torch
import datetime

import warnings
warnings.filterwarnings("ignore")

from Fires._utilities.utils_mlflow import load_model_from_mlflow
from Fires._utilities.utils_inference import get_cmip6_inference

In [2]:
config = munch.munchify(toml.load("/home/jovyan/work/codes/ML4Fires/config/cmip6_inference.toml"))
searfire_ds_path = "/home/jovyan/data/ML4Fires_data/data_100km.zarr"
seafire_ds = xr.open_zarr(searfire_ds_path)

In [3]:
import ipywidgets as widgets

scenario = widgets.Dropdown(
    options=[('SSP126', 'ssp126'), ('SSP245', 'ssp245'), ('SSP370', 'ssp370'),
             ('SSP585', 'ssp585')],
    value = 'ssp126',
    style={'description_width': '150px'}, 
    description='CMIP6 Scenario', disabled=False,)

year_range = widgets.IntRangeSlider(
    value=[2020, 2040],        # initial range
    min=2015,                 # min value
    max=2100,               # max value
    step=1,                # step size
    description='Year range:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

display(widgets.HBox([scenario, year_range]))


In [4]:
assert scenario.value != None, "Please select a CMIP6 scenario before proceesind"

In [5]:
# run_name=input()
run_name="last"
registered_model = load_model_from_mlflow(run_name, provenance=True)
# registered_model

Data from MLFlow downloaded in: /home/jovyan/work/codes/ML4Fires/MLFLOW/last


2025/05/09 14:34:30 WARNING mlflow.pytorch: Stored model version '2.4.1+cu121' does not match installed PyTorch version '2.4.1+cu124'


In [ ]:
predictions = get_cmip6_inference(
    seafire_ds=seafire_ds,
    run_name=run_name,
    scenario=scenario,
    year_range=year_range,
    config=config,
    model=registered_model)


Reading CMIP6 data for scenario ssp126 for year range 2020-2040
Loading the following CMIP6 data files...
lai: ['/home/jovyan/data/CMIP6/ScenarioMIP/CMCC/CMCC-ESM2/ssp126/r1i1p1f1/Eday/lai/gn/v20210126/lai_Eday_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20641231.nc']
tas: ['/home/jovyan/data/CMIP6/ScenarioMIP/CMCC/CMCC-ESM2/ssp126/r1i1p1f1/day/tas/gn/v20210126/tas_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20391231.nc', '/home/jovyan/data/CMIP6/ScenarioMIP/CMCC/CMCC-ESM2/ssp126/r1i1p1f1/day/tas/gn/v20210126/tas_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20400101-20641231.nc']
hur: ['/home/jovyan/data/CMIP6/ScenarioMIP/CMCC/CMCC-ESM2/ssp126/r1i1p1f1/day/hur/gn/v20210126/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20250101-20261231.nc', '/home/jovyan/data/CMIP6/ScenarioMIP/CMCC/CMCC-ESM2/ssp126/r1i1p1f1/day/hur/gn/v20210126/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20350101-20361231.nc', '/home/jovyan/data/CMIP6/ScenarioMIP/CMCC/CMCC-ESM2/ssp126/r1i1p1f1/day/hur/gn/v20210126/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_203

In [ ]:
from Fires._utilities.utils_inference import process_and_plot_cmip6infer, process_and_plot_data, load_input_data
input_data = load_input_data(searfire_ds_path, '2019', '2020') # Required by internal setting of variables in Fires._utilities.utils_inference

process_and_plot_cmip6infer(
	data=predictions["global_burned_areas"],
	temporal_aggregate_scheme=["sum","mean","mean"],
	label=f'Averaged burned area for {year_range.value[0]}-{year_range.value[1]} in scenario {scenario.value}',
	lats=seafire_ds.latitude.values,
	lons=seafire_ds.longitude.values,
    scale_min=0,
    scale_max=8000,
	model_name="Unet ++"
)

In [ ]:
predictions